# Univariate-marginal check on the top / bottom 20 ddE pairs

For every double-mutation pair in `{protein}/data/top_20_DRM.txt` and
`{protein}/data/bottom_20_DRM.txt` (IN, PR, RT):

* **top (synergistic) lists** — at least one of the two mutations must have a
  univariate marginal (mutant frequency in the MSA) above **0.5%**.
* **bottom (antagonistic) lists** — same 0.5% requirement on one mutation,
  **and** the *other* mutation must be **observed more than once** in the MSA
  (raw sequence count > 1).

The marginal used for the 0.5% test is the **weighted, reduced-alphabet**
frequency — the univariate marginal the Potts model is actually fit to. The
unweighted and full-alphabet (20-letter) frequencies are reported alongside so
it is easy to see whether the verdict is robust to that choice.

"Observed more than once" is an **unweighted raw count** — the number of MSA
sequences literally carrying that amino acid — since sequence weights are
reweighting, not observations. Reduced-alphabet counts are reported too.

In [1]:
import sys
import importlib
sys.path.append('../')  # repo root

import numpy as np
import pandas as pd
import utilities.functions as functions
importlib.reload(functions)

THRESHOLD = 0.005  # 0.5 % marginal, both lists
MIN_COUNT = 1      # bottom lists: the other mutation must be observed > MIN_COUNT times

## Load MSAs, weights and reduction dictionaries

In [2]:
IN_min_position, IN_max_position = 1, 263
PR_min_position, PR_max_position = 1, 99
RT_min_position, RT_max_position = 39, 226

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

IN_weights = functions.read_weights('IN/data/in.weights.txt')
PR_weights = functions.read_weights('PR/data/pr.exper.weights.txt')
RT_weights = functions.read_weights('RT/data/rt.weights.txt')

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux', 1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux', 0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux', 0)

proteins = {
    'IN': dict(seq=IN_all_seq, seq_unreduced=IN_all_seq_unreduced, weights=IN_weights,
               redux=IN_redux, min_pos=IN_min_position, max_pos=IN_max_position,
               top='IN/data/top_20_DRM.txt', bottom='IN/data/bottom_20_DRM.txt'),
    'PR': dict(seq=PR_all_seq, seq_unreduced=PR_all_seq_unreduced, weights=PR_weights,
               redux=PR_redux, min_pos=PR_min_position, max_pos=PR_max_position,
               top='PR/data/top_20_DRM.txt', bottom='PR/data/bottom_20_DRM.txt'),
    'RT': dict(seq=RT_all_seq, seq_unreduced=RT_all_seq_unreduced, weights=RT_weights,
               redux=RT_redux, min_pos=RT_min_position, max_pos=RT_max_position,
               top='RT/data/top_20_DRM.txt', bottom='RT/data/bottom_20_DRM.txt'),
}

for p, cfg in proteins.items():
    print(f"{p}: {len(cfg['seq'])} sequences, {len(cfg['weights'])} weights, "
          f"total weight {sum(cfg['weights']):.1f}, positions {cfg['min_pos']}-{cfg['max_pos']}")

IN: 1220 sequences, 1220 weights, total weight 993.0, positions 1-263
PR: 5710 sequences, 5710 weights, total weight 4650.0, positions 1-99
RT: 19194 sequences, 19194 weights, total weight 17130.0, positions 39-226


## Marginals and raw counts per mutation

In [3]:
# RT's DRM files are NRTI (first 20) followed by NNRTI (last 20)
RT_N_NRTI = 20


def drug_class(protein, idx):
    if protein != 'RT':
        return {'IN': 'INSTI', 'PR': 'PI'}[protein]
    return 'NRTI' if idx < RT_N_NRTI else 'NNRTI'


def marginals(mut, cfg):
    """Marginals and raw counts for one mutation.

    Returns (stats_dict, reduced_mutation). Values are NaN when the position
    falls outside the model's range, where the MSA columns are undefined."""
    nan = dict(f_reduced_weighted=np.nan, f_reduced=np.nan,
               f_unreduced_weighted=np.nan, f_unreduced=np.nan,
               n_reduced=np.nan, n_unreduced=np.nan)
    pos = functions.get_pos(mut)
    if not (cfg['min_pos'] <= pos <= cfg['max_pos']):
        return nan, None
    mut_red = functions.unreduced_to_reduced(cfg['redux'], mut)
    if mut_red[0] == '-' or mut_red[-1] == '-':
        return nan, mut_red
    _, _, f_red_w = functions.calculate_single_mutant_frequency_weighted(
        cfg['seq'], cfg['weights'], mut_red, cfg['min_pos'], cfg['max_pos'])
    n_red, _, f_red = functions.calculate_single_mutant_frequency(
        cfg['seq'], mut_red, cfg['min_pos'], cfg['max_pos'])
    _, _, f_unred_w = functions.calculate_single_mutant_frequency_weighted(
        cfg['seq_unreduced'], cfg['weights'], mut, cfg['min_pos'], cfg['max_pos'])
    n_unred, _, f_unred = functions.calculate_single_mutant_frequency(
        cfg['seq_unreduced'], mut, cfg['min_pos'], cfg['max_pos'])
    return dict(f_reduced_weighted=f_red_w, f_reduced=f_red,
                f_unreduced_weighted=f_unred_w, f_unreduced=f_unred,
                n_reduced=n_red, n_unreduced=n_unred), mut_red


def marginal_table(protein, cfg, which):
    rows = []
    pairs = functions.read_list_from_file(cfg[which])
    for idx, pair in enumerate(pairs):
        mut1, mut2 = [m.strip() for m in pair.split('-')]
        s1, red1 = marginals(mut1, cfg)
        s2, red2 = marginals(mut2, cfg)
        row = {
            'protein': protein,
            'drug_class': drug_class(protein, idx),
            'list': which,
            'rank': idx + 1,
            'pair': f'{mut1}-{mut2}',
            'mut1': mut1, 'mut2': mut2,
            'mut1_reduced': red1, 'mut2_reduced': red2,
            'in_model_range': red1 is not None and red2 is not None,
        }
        for k in s1:  # 'f_reduced' -> 'f1_reduced' / 'f2_reduced', same for n_*
            prefix, rest = k.split('_', 1)
            row[f'{prefix}1_{rest}'] = s1[k]
            row[f'{prefix}2_{rest}'] = s2[k]

        # --- 0.5 % marginal test (both lists) ---
        for tag in ['reduced_weighted', 'reduced', 'unreduced_weighted', 'unreduced']:
            a, b = row[f'f1_{tag}'], row[f'f2_{tag}']
            mx = np.nan if (np.isnan(a) and np.isnan(b)) else np.nanmax([a, b])
            row[f'max_{tag}'] = mx
            row[f'pass_marginal_{tag}'] = (mx > THRESHOLD) if not np.isnan(mx) else None

        # --- "other mutation observed more than once" (bottom lists only) ---
        # The partner of the >0.5 % mutation is the lower-marginal one, so the
        # requirement is equivalent to: the smaller raw count exceeds MIN_COUNT.
        for tag in ['unreduced', 'reduced']:
            a, b = row[f'n1_{tag}'], row[f'n2_{tag}']
            mn = np.nan if (np.isnan(a) and np.isnan(b)) else np.nanmin([a, b])
            row[f'min_count_{tag}'] = mn
            row[f'pass_count_{tag}'] = (mn > MIN_COUNT) if not np.isnan(mn) else None

        # --- overall verdict for this pair's list ---
        pm = row['pass_marginal_reduced_weighted']
        pc = row['pass_count_unreduced']
        row['pass'] = pm if which == 'top' else (
            None if (pm is None or pc is None) else bool(pm and pc))
        rows.append(row)
    return pd.DataFrame(rows)


tables = [marginal_table(p, cfg, which)
          for p, cfg in proteins.items() for which in ('top', 'bottom')]
df = pd.concat(tables, ignore_index=True)
print(f'{len(df)} pairs')
df.head()

160 pairs


,protein,drug_class,list,rank,pair,mut1,mut2,mut1_reduced,mut2_reduced,in_model_range,...,pass_marginal_reduced,max_unreduced_weighted,pass_marginal_unreduced_weighted,max_unreduced,pass_marginal_unreduced,min_count_unreduced,pass_count_unreduced,min_count_reduced,pass_count_reduced,pass
0,IN,INSTI,top,1,G140S-Q148H,G140S,Q148H,C140D,D148B,True,...,True,0.212890,True,0.199180,True,227,True,227,True,True
1,IN,INSTI,top,2,Y143C-S230R,Y143C,S230R,D143A,D230C,True,...,True,0.024404,True,0.023770,True,28,True,28,True,True
2,IN,INSTI,top,3,G140A-Q148K,G140A,Q148K,C140A,D148A,True,...,True,0.013427,True,0.011475,True,6,True,8,True,True
3,IN,INSTI,top,4,G140S-Q148R,G140S,Q148R,C140D,D148C,True,...,True,0.212890,True,0.199180,True,62,True,62,True,True
4,IN,INSTI,top,5,G140A-Q148R,G140A,Q148R,C140A,D148C,True,...,True,0.049734,True,0.050820,True,14,True,14,True,True


## Test results

In [4]:
summary = (df.groupby(['protein', 'drug_class', 'list'], sort=False)
             .agg(n_pairs=('pair', 'size'),
                  n_pass=('pass', lambda s: int((s == True).sum())),
                  n_fail=('pass', lambda s: int((s == False).sum())),
                  n_undefined=('pass', lambda s: int(s.isna().sum())),
                  min_of_max_marginal=('max_reduced_weighted', 'min'),
                  min_of_min_count=('min_count_unreduced', 'min'))
             .reset_index())
summary

,protein,drug_class,list,n_pairs,n_pass,n_fail,n_undefined,min_of_max_marginal,min_of_min_count
0,IN,INSTI,top,20,20,0,0,0.009651,1
1,IN,INSTI,bottom,20,20,0,0,0.013427,7
2,PR,PI,top,20,20,0,0,0.028846,9
3,PR,PI,bottom,20,20,0,0,0.044677,22
4,RT,NRTI,top,20,20,0,0,0.024203,12
5,RT,NNRTI,top,20,20,0,0,0.014541,26
6,RT,NRTI,bottom,20,20,0,0,0.058780,38
7,RT,NNRTI,bottom,20,20,0,0,0.049089,89


In [5]:
fails = df[df['pass'] == False]
if len(fails) == 0:
    print('PASS: every pair meets its list\'s criterion.')
else:
    print(f'{len(fails)} failing pair(s):')
    display(fails[['protein', 'drug_class', 'list', 'rank', 'pair',
                   'f1_reduced_weighted', 'f2_reduced_weighted', 'max_reduced_weighted',
                   'n1_unreduced', 'n2_unreduced', 'min_count_unreduced',
                   'pass_marginal_reduced_weighted', 'pass_count_unreduced']])

PASS: every pair meets its list's criterion.


In [6]:
# Break the bottom-list verdict into its two halves
bottom = df[df['list'] == 'bottom']
print('bottom lists only:')
print(f"  0.5% marginal failures : {int((bottom['pass_marginal_reduced_weighted'] == False).sum())}")
print(f"  'other observed >1' failures: {int((bottom['pass_count_unreduced'] == False).sum())}")
print()
print('Bottom pairs with the smallest raw count on the partner mutation:')
cols = ['protein', 'drug_class', 'pair', 'n1_unreduced', 'n2_unreduced',
        'min_count_unreduced', 'min_count_reduced', 'max_reduced_weighted', 'pass']
display(bottom.nsmallest(10, 'min_count_unreduced')[cols])

bottom lists only:
  0.5% marginal failures : 0
  'other observed >1' failures: 0

Bottom pairs with the smallest raw count on the partner mutation:


,protein,drug_class,pair,n1_unreduced,n2_unreduced,min_count_unreduced,min_count_reduced,max_reduced_weighted,pass
37,IN,INSTI,N155H-K160R,242,7,7,14,0.227843,True
36,IN,INSTI,N155H-V176L,242,10,10,10,0.227843,True
28,IN,INSTI,G140A-S147G,14,16,14,14,0.013427,True
33,IN,INSTI,G140A-S230N,14,119,14,14,0.096677,True
23,IN,INSTI,G140S-S147G,243,16,16,16,0.212890,True
30,IN,INSTI,N155H-G193D,242,16,16,17,0.227843,True
34,IN,INSTI,S147G-V151V,16,1100,16,16,0.892426,True
70,PR,PI,V82T-N83D,184,22,22,22,0.066968,True
38,IN,INSTI,Y143C-V234L,29,1071,29,29,0.879034,True
39,IN,INSTI,E92Q-G163R,38,32,32,40,0.043471,True


In [7]:
# Robustness: does either half of the test flip under a different marginal / alphabet?
for tag in ['reduced_weighted', 'reduced', 'unreduced_weighted', 'unreduced']:
    print(f'marginal {tag:22s} failing pairs (all lists): '
          f"{int((df[f'pass_marginal_{tag}'] == False).sum())}")
for tag in ['unreduced', 'reduced']:
    print(f'count    {tag:22s} failing pairs (bottom only): '
          f"{int((bottom[f'pass_count_{tag}'] == False).sum())}")

marginal reduced_weighted       failing pairs (all lists): 0
marginal reduced                failing pairs (all lists): 0
marginal unreduced_weighted     failing pairs (all lists): 0
marginal unreduced              failing pairs (all lists): 0
count    unreduced              failing pairs (bottom only): 0
count    reduced                failing pairs (bottom only): 0


## Write the CSV

In [8]:
df.to_csv('top_bottom_20_DRM_marginal_check.csv', index=False)
print(f'wrote top_bottom_20_DRM_marginal_check.csv  ({len(df)} pairs)')

wrote top_bottom_20_DRM_marginal_check.csv  (160 pairs)
